# Notebook 05 — Ensemble & Submission

**Requires**:
- Notebook 02 features parquet
- Notebook 03 Ridge predictions (`nlp_*.npy`)
- Notebook 04 Tree predictions (`lgb_*.npy` & `cb_*.npy`)

**Strategy**:
1. Load validation and test targets + model predictions.
2. Blend LightGBM and CatBoost predictions.
3. Apply **Multi-Level Dummy Fraction Scaling** to handle low-value target noise (placeholder labels entered by sellers).
4. Save final predictions to `../submissions/submission.csv`.

## 0. DATA MODE

In [1]:
# ============================================================
# DATA MODE: Changed to experiment mode
# ============================================================
DATA_MODE = "experiment"
DATA_PATHS = {
    "debug":      "../dataset/sampled/debug",
    "experiment": "../dataset/sampled/experiment",
    "full":       "../dataset"
}
DATA_DIR = DATA_PATHS[DATA_MODE]
print("Using dataset:", DATA_DIR)


Using dataset: ../dataset/sampled/experiment


## 1. Imports & Helpers

In [2]:
import os, numpy as np, pandas as pd

def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]))

print("✅ Imports & helpers done")


✅ Imports & helpers done


## 2. Load Predictions & Ground Truth

In [3]:
# Load data
df_test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
df_train_all = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

train_idx = np.load("../processed_features/train_indices.npy")
val_idx   = np.load("../processed_features/val_indices.npy")

df_train = df_train_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_train_all.iloc[val_idx].reset_index(drop=True)

y_train = df_train['PRODUCT_LENGTH'].values.astype(float)
y_val   = df_val['PRODUCT_LENGTH'].values.astype(float)

# Load predictions
lgb_va = np.load("../processed_features/lgb_val.npy")
lgb_te = np.load("../processed_features/lgb_test.npy")

cb_va  = np.load("../processed_features/cb_val.npy")
cb_te  = np.load("../processed_features/cb_test.npy")

print(f"Validation MAPEs:")
print(f"  LightGBM:  {mape(y_val, lgb_va)*100:.2f}%")
print(f"  CatBoost:  {mape(y_val, cb_va)*100:.2f}%")


Validation MAPEs:
  LightGBM:  92.54%
  CatBoost:  97.85%


## 3. Blend Models

In [4]:
# Blend weights from validation optimization (80% LightGBM + 20% CatBoost)
blend_va = np.clip(0.8 * lgb_va + 0.2 * cb_va, 0.5, None)
blend_te = np.clip(0.8 * lgb_te + 0.2 * cb_te, 0.5, None)

print(f"Blend Validation MAPE: {mape(y_val, blend_va)*100:.2f}%")


Blend Validation MAPE: 92.84%


## 4. Multi-Level Category Scaling (Dummy Mitigation)

This post-processing step corrects the over-prediction bias caused by placeholder targets (e.g. 1.0, 2.0, 1.0 cm entered by sellers) in certain product categories.

In [5]:
# Compute category dummy fractions from training data (y <= 100)
df_train['is_dummy'] = (y_train <= 100).astype(int)
dummy_fractions = df_train.groupby('PRODUCT_TYPE_ID')['is_dummy'].mean().to_dict()

def apply_dummy_scaling(preds, df_src):
    res = preds.copy()
    pids = df_src['PRODUCT_TYPE_ID'].values
    fracs = np.array([dummy_fractions.get(pid, 0.0) for pid in pids])
    
    # Apply optimized multi-level step scaling
    scale = np.ones(len(res))
    scale[fracs > 0.02] = 1.00
    scale[fracs > 0.05] = 0.40
    scale[fracs > 0.10] = 0.20
    scale[fracs > 0.20] = 0.10
    
    res = res * scale
    return np.clip(res, 0.5, None)

final_va = apply_dummy_scaling(blend_va, df_val)
final_te = apply_dummy_scaling(blend_te, df_test)

print(f"Validation MAPE before scaling: {mape(y_val, blend_va)*100:.2f}%")
print(f"Validation MAPE after scaling:  {mape(y_val, final_va)*100:.2f}%")


Validation MAPE before scaling: 92.84%
Validation MAPE after scaling:  86.67%


## 5. Generate Submission

In [6]:
submission = pd.DataFrame({
    'PRODUCT_ID':     df_test['PRODUCT_ID'].values,
    'PRODUCT_LENGTH': np.round(final_te, 2)
})

os.makedirs("../submissions", exist_ok=True)
submission.to_csv("../submissions/submission.csv", index=False)
print("Submission shape:", submission.shape)
print("✅ Saved to ../submissions/submission.csv!")
submission.head(10)


Submission shape: (50000, 2)
✅ Saved to ../submissions/submission.csv!


,PRODUCT_ID,PRODUCT_LENGTH
0,2794589,42.23
1,1735111,206.11
2,2138077,75.36
3,1414295,131.53
4,1836511,207.12
5,2584756,211.23
6,2981270,211.16
7,2107407,142.34
8,1039765,17.72
9,183804,211.95
